Avide lecteur de *L'Enracinement* de Simone Weil, et depuis peu creusant le sujet de la *Fin du village* (Le Goff), étant en outre bretonnant et curieux de comprendre « l'âme » des populations de mon Finistère natal, je me propose : 1° Télécharger les réponses de plusieurs paroisses finisteriennes en 1902 à une enquête diocésaine relative à la pratique du breton/compréhension de la langue française au catéchisme en 1902 2° D'effectuer une analyse HTR de l'écriture cursive dans le but de construire un dataframe 3° D'essayer de voir s'il y a une corrélation entre les votes à cette époque et cette enquête.

De la même manière, disposant de la liste électorale actuelle du Finistère (accessible légalement pour tout citoyen inscrit sur cette liste) et du registre des décès, je désirerais mesurer l'endogamie propre aux communes bretonnes (fait de naître, de vivre et de mourir dans la même commune ou proche) et son effet sur le vote.

In [ ]:
import requests
from bs4 import BeautifulSoup
import os
import time
import re
import unicodedata

# --- CONFIGURATION ---
BASE_URL = "https://bibliotheque.diocese-quimper.fr"
START_URL = "https://bibliotheque.diocese-quimper.fr/collections/show/89"
OUTPUT_DIR = "images_enquete_1902"

# Création du dossier de sauvegarde
if not os.path.exists(OUTPUT_DIR):
    os.makedirs(OUTPUT_DIR)

def nettoyer_nom_commune(texte):
    """
    Extrait le nom de la commune du titre.
    Ex: "Enquête sur le breton en 1902 : Taulé" -> "Taule"
    """
    # On garde ce qu'il y a après les " : " s'ils existent
    if ":" in texte:
        nom = texte.split(":")[-1].strip()
    else:
        nom = texte.strip()
    
    # Suppression des accents et caractères spéciaux pour le nom de fichier
    nom = unicodedata.normalize('NFKD', nom).encode('ASCII', 'ignore').decode('utf-8')
    nom = re.sub(r'[^\w\s-]', '', nom).strip().replace(' ', '-')
    return nom.upper()

def trouver_code_insee(nom_commune):
    """
    ATTENTION : Le site ne fournit pas le code INSEE.
    Ceci est une fonction placeholder. 
    Pour avoir les vrais codes, il faudrait un dictionnaire de mapping ou une API externe.
    """
    # Exemple de dictionnaire manuel (à compléter si besoin)
    mapping = {
        "TAULE": "29279",
        "QUIMPER": "29232",
        # Ajoutez d'autres villes ici si vous les connaissez
    }
    return mapping.get(nom_commune, "00000") # 00000 par défaut si inconnu

def telecharger_image(url_item):
    try:
        response = requests.get(url_item)
        soup = BeautifulSoup(response.content, 'html.parser')

        # 1. Récupération du Titre (Commune)
        titre_h1 = soup.find('h1')
        if not titre_h1:
            print(f"[ERREUR] Pas de titre trouvé sur {url_item}")
            return
        
        nom_brut = titre_h1.text.strip()
        nom_commune = nettoyer_nom_commune(nom_brut)
        code_insee = trouver_code_insee(nom_commune)

        # 2. Récupération du lien de l'image originale
        # Dans Omeka, les images sont souvent dans une div 'item-images' ou 'element-text'
        # On cherche le lien qui mène au fichier original (souvent 'original' dans l'url ou class 'download-file')
        image_div = soup.find('div', id='item-images')
        
        img_url = None
        if image_div:
            link_tag = image_div.find('a')
            if link_tag and 'href' in link_tag.attrs:
                img_url = link_tag['href']
        
        # Si lien non absolu, on ajoute le domaine
        if img_url and not img_url.startswith('http'):
            img_url = img_url # Souvent absolu sur Omeka, mais à vérifier

        if not img_url:
            print(f"[PAS D'IMAGE] Ignoré : {nom_commune}")
            return

        # 3. Téléchargement et Renommage
        # Format demandé : enquete_breton_1902_NOMCOMMUNE_codeinsee
        extension = img_url.split('.')[-1]
        nom_fichier = f"enquete_breton_1902_{nom_commune}_{code_insee}.{extension}"
        chemin_complet = os.path.join(OUTPUT_DIR, nom_fichier)

        # On télécharge le fichier
        img_data = requests.get(img_url).content
        with open(chemin_complet, 'wb') as handler:
            handler.write(img_data)
        
        print(f"[OK] Téléchargé : {nom_fichier}")

    except Exception as e:
        print(f"[ERREUR] Problème sur {url_item} : {e}")

def main():
    current_url = START_URL
    page_count = 1

    while current_url:
        print(f"--- Traitement de la page {page_count} ---")
        response = requests.get(current_url)
        soup = BeautifulSoup(response.content, 'html.parser')

        # Trouver tous les items de la liste (class="item hentry")
        items = soup.find_all(class_="item")
        
        for item in items:
            # Trouver le lien vers la page détaillée de l'item
            link_tag = item.find('a', class_='permalink') # Souvent permalink dans Omeka
            if not link_tag:
                h2 = item.find('h2')
                link_tag = h2.find('a') if h2 else item.find('a')
            
            if link_tag and 'href' in link_tag.attrs:
                full_item_url = link_tag['href']
                if not full_item_url.startswith('http'):
                    full_item_url = BASE_URL + full_item_url
                
                telecharger_image(full_item_url)
                time.sleep(0.5) # Pause pour être gentil avec le serveur

        # Gestion de la pagination (Lien "Suivant")
        next_page_link = soup.find('a', class_='next') # Classique Omeka
        # Parfois c'est dans une liste de pagination
        if not next_page_link:
            pagination = soup.find('ul', class_='pagination')
            if pagination:
                next_li = pagination.find('li', class_='pagination_next')
                if next_li:
                    next_page_link = next_li.find('a')

        if next_page_link and 'href' in next_page_link.attrs:
            next_url = next_page_link['href']
            if not next_url.startswith('http'):
                current_url = BASE_URL + next_url
            else:
                current_url = next_url
            page_count += 1
        else:
            current_url = None

    print("\nTerminé ! Vérifiez le dossier 'images_enquete_1902'.")

if __name__ == "__main__":
    main()